# 🚗 Árbol de regresión — Caso MPG

## 📖 Dónde estamos
Llevas dos modelos sobre estos mismos autos:

| Modelo | R² en prueba | RMSE |
|---|---|---|
| Regresión simple (`weight`) | 0.653 | 4.21 |
| Regresión múltiple (todas) | 0.792 | 3.26 |
| Regresión lineal (RFE) | 0.794 | 3.24 |

Todos esos modelos asumen que la relación es una **recta**. Hoy probamos uno que no asume ninguna forma y que, además, **se puede dibujar y leer**.

> ### ❓ ¿Le gana un árbol a la regresión lineal, y qué nos cuenta que la recta no podía contar?

## ⏱️ Agenda de la sesión (3 horas)
| Parte | Dónde | Tiempo |
|---|---|---|
| Fundamento: cómo funciona un árbol | `fundamentos.ipynb` | 30 min |
| **Caso MPG — árbol de regresión** | `caso_mpg.ipynb` | 50 min |
| ☕ Descanso | | 10 min |
| **Caso Titanic — árbol de clasificación** | `caso_titanic.ipynb` | 45 min |
| Cierre del curso: los cuatro modelos | `caso_titanic.ipynb` | 30 min |

---
# 0️⃣ Preparación

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_frequencies(df, columnas, top_n=None)` | Frecuencia de variables categóricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_simple_regression(x, y, results)` | Recta ajustada de un modelo OLS con 1 variable |
| `plot_actual_vs_predicted(y_real, y_pred)` | Valores reales vs predichos |
| `plot_residuals(y_real, y_pred)` | Residuales vs predichos |
| `plot_rfecv(rfecv)` | R² según el número de variables seleccionadas por RFECV |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas',
        labels={col: col.capitalize() for col in columns}
    )
    fig.update_layout(width=1200, height=1200, title_font_size=20)
    fig.update_traces(diagonal_visible=True)
    fig.show()


def plot_simple_regression(x, y, results):
    """Dispersión de una variable vs el objetivo con la recta ajustada por un OLS de 1 variable."""
    b0, b1 = results.params.iloc[0], results.params.iloc[1]
    x_name = getattr(x, 'name', None) or 'x'
    y_name = getattr(y, 'name', None) or 'y'
    x_line = np.linspace(np.min(x), np.max(x), 100)

    fig = px.scatter(
        x=np.asarray(x),
        y=np.asarray(y),
        opacity=0.6,
        labels={'x': x_name, 'y': y_name},
        title=f'{y_name} = {b0:.2f} + ({b1:.4f}) · {x_name}',
        template='plotly_white'
    )
    fig.add_trace(go.Scatter(
        x=x_line,
        y=b0 + b1 * x_line,
        mode='lines',
        name='Recta OLS',
        line=dict(color='red', width=3)
    ))
    fig.show()


def plot_actual_vs_predicted(y_true, y_pred, title='Real vs Predicho'):
    """Valores reales vs predichos; un modelo perfecto cae sobre la diagonal roja."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())

    fig = px.scatter(
        x=y_true,
        y=y_pred,
        opacity=0.5,
        labels={'x': 'Valor real', 'y': 'Valor predicho'},
        title=title,
        template='plotly_white'
    )
    fig.add_shape(
        type='line', x0=lo, y0=lo, x1=hi, y1=hi,
        line=dict(color='red', dash='dash')
    )
    fig.show()


def plot_residuals(y_true, y_pred, title='Residuales vs Predicho'):
    """Residuales vs predichos; buscamos una nube sin patrón alrededor de 0."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)

    fig = px.scatter(
        x=y_pred,
        y=y_true - y_pred,
        opacity=0.5,
        labels={'x': 'Valor predicho', 'y': 'Residual (real − predicho)'},
        title=title,
        template='plotly_white'
    )
    fig.add_hline(y=0, line_dash='dash', line_color='red')
    fig.show()


def plot_rfecv(rfecv):
    """R² promedio de validación cruzada según el número de variables que conserva RFECV."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=rfecv.cv_results_['n_features'],
        y=rfecv.cv_results_['mean_test_score'],
        mode='lines+markers',
        line=dict(color='steelblue', width=3),
        marker=dict(size=7),
        name='R² promedio (CV)'
    ))
    fig.update_layout(
        title='RFECV — R² según número de variables seleccionadas',
        xaxis_title='Número de variables',
        yaxis_title='R² (validación cruzada)',
        template='plotly_white',
        width=900, height=450
    )
    fig.show()

### 🧰 Funciones de machine learning

Para los modelos de scikit-learn:

| Función | ¿Para qué sirve? |
|---|---|
| `escalar(X_train, X_test)` | Estandariza ambos conjuntos ajustando sólo con train |
| `evaluar_regresion(nombre, y_real, y_pred)` | R², RMSE y MAE |
| `plot_curva_k(ks, scores, metrica)` | Desempeño de validación cruzada para cada k, y regresa el mejor |
| `plot_curva_complejidad(valores, train, test, parametro, metrica)` | Entrenamiento vs prueba: así se ve el sobreajuste |
| `plot_importancias(modelo, columnas)` | Importancia de variables de un árbol |
| `plot_arbol(modelo, columnas, max_depth)` | Dibuja las reglas del árbol |

In [ ]:
# Funciones de machine learning: evaluar y diagnosticar modelos de scikit-learn
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import plot_tree


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE y MAE de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | MAE {metricas['MAE']}")
    return metricas


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    from sklearn.preprocessing import StandardScaler

    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def plot_curva_k(k_values, scores, metrica='RMSE', menor_es_mejor=True):
    """Desempeño de validación cruzada para cada valor de k en KNN."""
    k_values, scores = list(k_values), list(scores)
    mejor = k_values[int(np.argmin(scores) if menor_es_mejor else np.argmax(scores))]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=k_values, y=scores, mode='lines+markers',
        line=dict(color='steelblue', width=3), marker=dict(size=7), name=f'{metrica} (CV)'
    ))
    fig.add_vline(x=mejor, line_dash='dash', line_color='red', annotation_text=f'mejor k = {mejor}')
    fig.update_layout(
        title=f'Elección de k — {metrica} de validación cruzada',
        xaxis_title='k (número de vecinos)', yaxis_title=f'{metrica} (validación cruzada)',
        template='plotly_white', width=900, height=450
    )
    fig.show()
    return mejor


def plot_curva_complejidad(valores, score_train, score_test, parametro='max_depth', metrica='R²'):
    """Desempeño en entrenamiento vs prueba al aumentar la complejidad: así se ve el sobreajuste."""
    datos = pd.DataFrame({parametro: list(valores) * 2,
                          metrica: list(score_train) + list(score_test),
                          'conjunto': ['Entrenamiento'] * len(score_train) + ['Prueba'] * len(score_test)})

    fig = px.line(
        datos, x=parametro, y=metrica, color='conjunto', markers=True,
        title=f'Sobreajuste: {metrica} según {parametro}', template='plotly_white',
        color_discrete_map={'Entrenamiento': 'steelblue', 'Prueba': 'tomato'}
    )
    fig.update_layout(width=900, height=450)
    fig.show()


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)


def plot_arbol(modelo, columnas, max_depth=3, class_names=None):
    """Dibuja los primeros niveles del árbol para poder leer sus reglas."""
    fig, ax = plt.subplots(figsize=(22, 9))
    plot_tree(
        modelo, max_depth=max_depth, feature_names=list(columnas), class_names=class_names,
        filled=True, rounded=True, fontsize=9, impurity=False, ax=ax
    )
    ax.set_title(f'Árbol de decisión — primeros {max_depth} niveles', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 📥 Los datos, ya preparados

Este caso lo trabajaste en la sesión de regresión lineal, así que no repetimos la exploración ni el preprocesamiento: la celda siguiente trae las dos funciones que hacen lo mismo que hiciste a mano.

| Función | ¿Qué hace? |
|---|---|
| `cargar_mpg()` | Descarga la base SQLite y elimina los nulos de `horsepower` |
| `preparar_mpg(df)` | Separa X/y, hace el split 80/20 (`random_state=42`) y codifica `origin` con los autos americanos como referencia |

Usa exactamente la **misma partición** de aquella sesión, así que las métricas son comparables.

In [ ]:
# Datos del caso MPG: carga y preparación (idénticas a las de la sesión 1)
import pandas as pd
import requests
import sqlite3
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder


def cargar_mpg():
    """Descarga la base SQLite del caso MPG y regresa el DataFrame sin nulos (392 autos)."""
    url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/mpg.db"
    with open("mpg.db", "wb") as f:
        f.write(requests.get(url).content)

    query = """
    SELECT
        O.mpg,
        O.cylinders,
        O.displacement,
        O.horsepower,
        O.weight,
        O.acceleration,
        O.model_year,
        ORG.origin,
        N.name
    FROM
        Observation AS O
    JOIN
        Origin AS ORG ON O.origin_id = ORG.origin_id
    JOIN
        Name AS N ON O.name_id = N.name_id
    """
    df = pd.read_sql_query(query, sqlite3.connect("mpg.db"))

    # Sólo 'horsepower' tiene nulos (1.5% de las filas): se eliminan
    return df.dropna(subset=['horsepower'])


def preparar_mpg(df, test_size=0.2, random_state=42):
    """Split 80/20 y One-Hot de 'origin' con los autos americanos como referencia.

    Regresa X_train, X_test, y_train, y_test listos para modelar (todo numérico).
    """
    X = df.drop(columns=['mpg', 'name'])
    y = df['mpg']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    ohe = OneHotEncoder(drop=['usa'], sparse_output=False)
    dummies_train = pd.DataFrame(
        ohe.fit_transform(X_train[['origin']]),
        columns=ohe.get_feature_names_out(), index=X_train.index
    )
    dummies_test = pd.DataFrame(
        ohe.transform(X_test[['origin']]),
        columns=ohe.get_feature_names_out(), index=X_test.index
    )

    X_train_enc = pd.concat([X_train.drop(columns=['origin']), dummies_train], axis=1)
    X_test_enc = pd.concat([X_test.drop(columns=['origin']), dummies_test], axis=1)

    print(f"Train: {X_train_enc.shape} | Test: {X_test_enc.shape}")
    print(f"Variables: {list(X_train_enc.columns)}")
    return X_train_enc, X_test_enc, y_train, y_test

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
df = cargar_mpg()
X_train, X_test, y_train, y_test = preparar_mpg(df)

Misma partición de siempre. Y fíjate en lo que **no** vamos a hacer hoy: **escalar**. El árbol no mide distancias, así que no lo necesita.

---
# 1️⃣ Un primer árbol, sin podar (15 min)

Empecemos por el error clásico: dejarlo crecer sin límite.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

# 1. Ajusta un árbol SIN límites: DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
# 2. comparacion = []  y agrega evaluar_regresion('Árbol sin podar', y_test, <predicciones en test>)
# 3. Imprime también su R² en ENTRENAMIENTO: arbol_sin_podar.score(X_train, y_train)
# 4. Y cuántas hojas tiene: .get_n_leaves()
# ✍️ Tu código aquí


✍️ **Compara los dos números:**
- ¿Qué R² obtuvo en entrenamiento? ¿Y en prueba?
- ¿Cuántas hojas creó, y cuántos autos hay en entrenamiento?
- ¿Cómo se llama lo que está pasando cuando un modelo acierta casi todo en entrenamiento y falla en prueba?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ Podar con validación cruzada (20 min)

Buscamos la combinación de `max_depth` (qué tan profundo) y `min_samples_leaf` (cuántos autos mínimo por hoja) que da el menor RMSE en validación cruzada. `GridSearchCV` prueba todas las combinaciones por nosotros.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.model_selection import GridSearchCV, KFold

parametros = {
    'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

busqueda = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    parametros,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)
busqueda.fit(X_train, y_train)

arbol = busqueda.best_estimator_

print(f"Mejores parámetros: {busqueda.best_params_}")
print(f"RMSE de validación cruzada: {-busqueda.best_score_:.3f}")

In [ ]:
# Evalúa el árbol podado en prueba, agrégalo a comparacion, e imprime su R² en
# entrenamiento y su número de hojas para compararlos con los del árbol sin podar
# ✍️ Tu código aquí


✍️ **Llena esta tabla con tus resultados:**

| | Sin podar | Podado |
|---|---|---|
| R² entrenamiento | | |
| R² prueba | | |
| Hojas | | |

¿Qué pasó con la distancia entre entrenamiento y prueba? ¿Qué significa eso?

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ Ver el sobreajuste en una gráfica (10 min)

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15]
r2_train, r2_test = [], []

for d in profundidades:
    modelo = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    r2_train.append(modelo.score(X_train, y_train))
    r2_test.append(modelo.score(X_test, y_test))

plot_curva_complejidad(profundidades, r2_train, r2_test, parametro='max_depth', metrica='R²')

✍️ ¿En qué profundidad se separan las dos curvas? ¿Coincide con la que eligió la validación cruzada?

RELLENAR CON TUS COMENTARIOS

---
# 4️⃣ Lo que el árbol sí regala: leerlo (15 min)

In [ ]:
# Dibuja los primeros 3 niveles del árbol
# Llama a plot_arbol(arbol, X_train.columns, max_depth=3)
# ✍️ Tu código aquí


Cada caja dice: la pregunta, cuántos autos llegaron ahí (`samples`) y qué predice (`value`). Síguelo de arriba hacia abajo y tienes el razonamiento completo del modelo, caso por caso.

In [ ]:
# Grafica la importancia de las variables
# Llama a plot_importancias(arbol, X_train.columns)
# ✍️ Tu código aquí


✍️ **Una sorpresa que vale la pena entender:**
- ¿Qué variable se llevó la mayor importancia? ¿Es la misma que la regresión lineal eligió como estrella en la sesión 1?
- Calcula `df[['displacement', 'weight']].corr()`. Con ese número, ¿cómo explicas que una tenga importancia alta y la otra casi cero?
- ¿Dirías que la variable con importancia baja "no sirve"? Justifica.

RELLENAR CON TUS COMENTARIOS

## 📝 Cierre del caso
✍️ Compara los dos modelos que llevas sobre MPG (regresión lineal y árbol): ¿cuál predice mejor?, ¿cuál explica mejor?, ¿cuál escogerías y por qué?

Ahora el caso de clasificación: `caso_titanic.ipynb`.